# Random Forest Submission V4
This notebook mirrors the latest RF V4 pipeline and aligns its structure with the newest XGBoost notebook.

## 1. Setup and Shared Dataset Discovery
Locate the project root and load the aligned Chicago/Texas shared datasets without hardcoding machine-specific paths.

In [ ]:
from __future__ import annotations


from pathlib import Path
from typing import Dict, List, Tuple

import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit


def resolve_this_dir() -> Path:
    if "__file__" in globals():
        return Path(__file__).resolve().parent

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == "code" and (candidate.parent / "outputs").exists():
            return candidate
        if (candidate / "RF_FINAL_SUBMISSION_V4.py").exists():
            return candidate
    return cwd


THIS_DIR = resolve_this_dir()
PACKAGE_DIR = THIS_DIR.parent
OUTPUT_DIR = PACKAGE_DIR / "outputs"

TARGETS = [
    ("label_THEFT", "THEFT"),
    ("label_BATTERY", "BATTERY"),
    ("label_CRIMINAL_DAMAGE", "CRIMINAL DAMAGE"),
]
TASK_NAME_BY_TARGET = dict(TARGETS)
TASK_SLUG_BY_TARGET = {
    "label_THEFT": "THEFT",
    "label_BATTERY": "BATTERY",
    "label_CRIMINAL_DAMAGE": "CRIMINAL_DAMAGE",
}
RISK_GROUP_ORDER = ["Low Risk", "Medium Risk", "High Risk"]
QUARTER_ORDER = ["Q1", "Q2", "Q3", "Q4"]
HOTSPOT_K_LIST = [5, 10, 20]

RF_PARAMS = {
    "n_estimators": 100,
    "max_depth": 15,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
}


def iter_nearby_roots() -> List[Path]:
    """Return nearby candidate roots around this file and the current working directory."""
    roots: List[Path] = []
    seen = set()
    for start in [THIS_DIR, Path.cwd().resolve()]:
        for candidate in [start, *start.parents]:
            if candidate not in seen:
                seen.add(candidate)
                roots.append(candidate)
    return roots


def find_workspace_root() -> Path:
    """Locate the project root by finding the expected sibling work folders."""
    for root in iter_nearby_roots():
        if (root / "my work").exists() and (root / "teammates work").exists():
            return root
    raise FileNotFoundError(
        "Could not locate the project root. Expected sibling folders named "
        "'my work' and 'teammates work' near this script."
    )


WORKSPACE_ROOT = find_workspace_root()
DATA_ROOT = WORKSPACE_ROOT / "teammates work" / "Unified Data Set Processing"


def resolve_data_file(folder_name: str, file_name: str) -> Path:
    """Locate a shared dataset without hardcoding one absolute path."""
    direct_path = DATA_ROOT / folder_name / file_name
    if direct_path.exists():
        return direct_path

    for root in iter_nearby_roots():
        for match in root.rglob(file_name):
            if folder_name.replace("\\", "/") in match.as_posix():
                return match

    raise FileNotFoundError(f"Could not find dataset '{file_name}' under a nearby '{folder_name}' directory.")


CHICAGO_TRAIN_X = resolve_data_file("Chicago-Standardized Dataset", "chicago_2015_2024_X_shared_42cols.csv")
CHICAGO_TRAIN_FULL = resolve_data_file("Chicago-Standardized Dataset", "chicago_2015_2024_shared_final.csv")
CHICAGO_TEST_X = resolve_data_file("Chicago-Standardized Dataset", "chicago_2025_X_shared_42cols.csv")
CHICAGO_TEST_FULL = resolve_data_file("Chicago-Standardized Dataset", "chicago_2025_shared_final.csv")
TEXAS_X = resolve_data_file("Texas-Standard Dataset", "tx_external_X_shared_42cols.csv")
TEXAS_FULL = resolve_data_file("Texas-Standard Dataset", "tx_external_shared_final_42cols.csv")


## 2. Metric Helpers
Define overall metrics and group-level metrics so the RF outputs can follow the same comparison structure as the latest XGBoost notebook.

In [ ]:
def safe_auc(y_true: pd.Series, y_prob: pd.Series) -> float:
    """Guard against groups that contain only one class."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            return float(roc_auc_score(y_true, y_prob))
        except ValueError:
            return float("nan")


def evaluate_binary_classification(y_true: pd.Series, y_pred: pd.Series, y_prob: pd.Series, dataset_name: str) -> Dict[str, object]:
    """Compute the standard summary metrics used in the write-up."""
    return {
        "dataset": dataset_name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc": safe_auc(y_true, y_prob),
    }


def safe_group_metrics(group: pd.DataFrame) -> pd.Series:
    """Compute accuracy metrics for one spatial or temporal slice."""
    return pd.Series(
        {
            "n_samples": int(len(group)),
            "positive_rate": float(group["y_true"].mean()),
            "accuracy": float(accuracy_score(group["y_true"], group["y_pred"])),
            "precision": float(precision_score(group["y_true"], group["y_pred"], zero_division=0)),
            "recall": float(recall_score(group["y_true"], group["y_pred"], zero_division=0)),
            "f1": float(f1_score(group["y_true"], group["y_pred"], zero_division=0)),
            "roc_auc": safe_auc(group["y_true"], group["y_prob"]),
        }
    )


def get_texas_spatial_unit_col(df: pd.DataFrame) -> str:
    """Use the same spatial-unit detection logic as the teammate XGBoost notebook."""
    if "grid_id" in df.columns:
        return "grid_id"
    if "agency_id" in df.columns:
        return "agency_id"
    raise ValueError("Texas external dataset has neither 'grid_id' nor 'agency_id'.")


def build_eval_df_for_target(
    unit_col: str,
    target_col: str,
    task_name: str,
    model: RandomForestClassifier,
    full_df: pd.DataFrame,
    feature_df: pd.DataFrame,
) -> pd.DataFrame:
    """Build a prediction table that can be reused across later analyses."""
    eval_df = full_df[[unit_col, "iso_year", "iso_week", "year_week"]].copy()
    eval_df["y_true"] = full_df[target_col].values
    eval_df["y_pred"] = model.predict(feature_df)
    eval_df["y_prob"] = model.predict_proba(feature_df)[:, 1]
    eval_df["task"] = task_name
    return eval_df


## 3. Training and TimeSeriesSplit CV
Sort rows by `iso_year`, `iso_week`, and `grid_id`, then run 5-fold TimeSeriesSplit to match the team CV protocol.

In [ ]:
def align_train_for_timeseries(train_full: pd.DataFrame, train_X: pd.DataFrame, target_col: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Match the latest XGBoost notebook:
    sort by iso_year, iso_week, and grid_id, then reorder X accordingly.
    """
    train_sorted = train_full.copy()
    train_sorted["_row_id_for_alignment"] = np.arange(len(train_sorted))
    train_sorted = train_sorted.sort_values(
        by=["iso_year", "iso_week", "grid_id"],
        ascending=[True, True, True],
    ).reset_index(drop=True)

    X_train_sorted = train_X.iloc[train_sorted["_row_id_for_alignment"]].reset_index(drop=True)
    y_train_sorted = train_sorted[target_col].copy()
    return train_sorted, X_train_sorted, y_train_sorted


def run_timeseries_cv_for_target(train_full: pd.DataFrame, train_X: pd.DataFrame, target_col: str, task_name: str, n_splits: int = 5) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run 5-fold TimeSeriesSplit CV aligned with XGBoost and MLP."""
    _train_sorted, X_train_sorted, y_train_sorted = align_train_for_timeseries(train_full, train_X, target_col)
    tscv = TimeSeriesSplit(n_splits=n_splits)

    cv_results: List[Dict[str, object]] = []
    fold_num = 1
    for train_idx, val_idx in tscv.split(X_train_sorted):
        X_tr = X_train_sorted.iloc[train_idx]
        X_val = X_train_sorted.iloc[val_idx]
        y_tr = y_train_sorted.iloc[train_idx]
        y_val = y_train_sorted.iloc[val_idx]

        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X_tr, y_tr)

        y_val_pred = model.predict(X_val)
        y_val_prob = model.predict_proba(X_val)[:, 1]

        cv_results.append(
            {
                "target_col": target_col,
                "task": task_name,
                "fold": fold_num,
                "accuracy": float(accuracy_score(y_val, y_val_pred)),
                "precision": float(precision_score(y_val, y_val_pred, zero_division=0)),
                "recall": float(recall_score(y_val, y_val_pred, zero_division=0)),
                "f1": float(f1_score(y_val, y_val_pred, zero_division=0)),
                "auc": safe_auc(y_val, pd.Series(y_val_prob)),
            }
        )
        fold_num += 1

    cv_results_df = pd.DataFrame(cv_results)
    cv_summary_df = pd.DataFrame(
        {
            "target_col": [target_col] * 5,
            "task": [task_name] * 5,
            "metric": ["accuracy", "precision", "recall", "f1", "auc"],
            "mean": [
                float(cv_results_df["accuracy"].mean()),
                float(cv_results_df["precision"].mean()),
                float(cv_results_df["recall"].mean()),
                float(cv_results_df["f1"].mean()),
                float(cv_results_df["auc"].mean()),
            ],
            "std": [
                float(cv_results_df["accuracy"].std(ddof=1)),
                float(cv_results_df["precision"].std(ddof=1)),
                float(cv_results_df["recall"].std(ddof=1)),
                float(cv_results_df["f1"].std(ddof=1)),
                float(cv_results_df["auc"].std(ddof=1)),
            ],
        }
    )
    return cv_results_df, cv_summary_df


def run_rf_for_one_target(
    target_col: str,
    task_name: str,
    train_full: pd.DataFrame,
    train_X: pd.DataFrame,
    test_full: pd.DataFrame,
    test_X: pd.DataFrame,
    ext_full: pd.DataFrame,
    ext_X: pd.DataFrame,
) -> Dict[str, object]:
    """Fit the final RF model and evaluate Chicago 2025 plus Texas external data."""
    y_train = train_full[target_col].copy()
    y_test = test_full[target_col].copy()
    y_ext = ext_full[target_col].copy()

    model = RandomForestClassifier(**RF_PARAMS)
    model.fit(train_X, y_train)

    y_test_pred = model.predict(test_X)
    y_test_prob = model.predict_proba(test_X)[:, 1]
    chicago_result = evaluate_binary_classification(y_test, y_test_pred, y_test_prob, "Chicago 2025")

    y_ext_pred = model.predict(ext_X)
    y_ext_prob = model.predict_proba(ext_X)[:, 1]
    texas_result = evaluate_binary_classification(y_ext, y_ext_pred, y_ext_prob, "Texas External")

    return {
        "target_col": target_col,
        "task": task_name,
        "model": model,
        "chicago_result": chicago_result,
        "texas_result": texas_result,
        "y_test_pred": y_test_pred,
        "y_test_prob": y_test_prob,
        "y_ext_pred": y_ext_pred,
        "y_ext_prob": y_ext_prob,
    }


## 4. Spatial and Temporal Evaluation Helpers
Implement Chicago spatial risk-group evaluation, quarterly temporal stability, hotspot hit-rate / coverage, spatial error analysis, and Texas spatial generalization.

In [ ]:
def run_spatial_evaluation_for_target(
    target_col: str,
    task_name: str,
    model: RandomForestClassifier,
    train_full: pd.DataFrame,
    test_full: pd.DataFrame,
    test_X: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Replicate the teammate XGBoost spatial evaluation pipeline on Chicago 2025."""
    chicago_eval_df = build_eval_df_for_target("grid_id", target_col, task_name, model, test_full, test_X)

    spatial_eval_df = (
        chicago_eval_df.groupby("grid_id")[["y_true", "y_pred", "y_prob"]]
        .apply(safe_group_metrics)
        .reset_index()
    )
    spatial_eval_filtered = spatial_eval_df[spatial_eval_df["n_samples"] >= 10].copy()

    train_grid_risk = (
        train_full.groupby("grid_id")[target_col]
        .mean()
        .reset_index()
        .rename(columns={target_col: "historical_risk"})
    )

    spatial_eval_with_risk = spatial_eval_filtered.merge(train_grid_risk, on="grid_id", how="left")
    spatial_eval_with_risk["risk_group"] = pd.qcut(
        spatial_eval_with_risk["historical_risk"].rank(method="first"),
        q=3,
        labels=RISK_GROUP_ORDER,
    )

    risk_group_summary = (
        spatial_eval_with_risk.groupby("risk_group", observed=False)[["accuracy", "precision", "recall", "f1", "roc_auc"]]
        .mean()
        .reset_index()
    )
    risk_group_summary["task"] = task_name

    chicago_eval_df["task"] = task_name
    spatial_eval_df["task"] = task_name
    spatial_eval_with_risk["task"] = task_name
    return chicago_eval_df, spatial_eval_df, spatial_eval_with_risk, risk_group_summary


def run_temporal_evaluation_for_target(
    target_col: str,
    task_name: str,
    model: RandomForestClassifier,
    test_full: pd.DataFrame,
    test_X: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Replicate the teammate XGBoost weekly and quarterly temporal evaluation."""
    chicago_eval_df = build_eval_df_for_target("grid_id", target_col, task_name, model, test_full, test_X)

    temporal_eval_df = (
        chicago_eval_df.groupby("iso_week")[["y_true", "y_pred", "y_prob"]]
        .apply(safe_group_metrics)
        .reset_index()
    )
    temporal_eval_df["task"] = task_name

    chicago_eval_df = chicago_eval_df.copy()
    chicago_eval_df["quarter"] = pd.cut(
        chicago_eval_df["iso_week"],
        bins=[0, 13, 26, 39, 53],
        labels=QUARTER_ORDER,
    )
    quarter_eval_df = (
        chicago_eval_df.groupby("quarter", observed=False)[["y_true", "y_pred", "y_prob"]]
        .apply(safe_group_metrics)
        .reset_index()
    )
    quarter_eval_df["task"] = task_name
    return temporal_eval_df, quarter_eval_df


def compute_hotspot_metrics(eval_df: pd.DataFrame, unit_col: str, k_list: List[int]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Match the XGBoost hotspot hit-rate and coverage calculation."""
    grid_summary = (
        eval_df.groupby(unit_col)
        .agg(
            true_count=("y_true", "sum"),
            pred_count=("y_pred", "sum"),
            pred_score=("y_prob", "sum"),
            n_samples=("y_true", "size"),
        )
        .reset_index()
    )

    results = []
    total_true_events = float(grid_summary["true_count"].sum())
    for k in k_list:
        true_topk = set(grid_summary.sort_values("true_count", ascending=False).head(k)[unit_col])
        pred_topk_df = grid_summary.sort_values("pred_score", ascending=False).head(k)
        hit_count = len(true_topk & set(pred_topk_df[unit_col]))
        hit_rate = float(hit_count / k) if k > 0 else float("nan")
        covered_true_events = float(pred_topk_df["true_count"].sum())
        coverage = float(covered_true_events / total_true_events) if total_true_events > 0 else float("nan")

        results.append(
            {
                "k": int(k),
                "hit_count": int(hit_count),
                "hit_rate": hit_rate,
                "coverage": coverage,
            }
        )

    return grid_summary, pd.DataFrame(results)


def compute_spatial_error_by_unit(eval_df: pd.DataFrame, unit_col: str) -> pd.DataFrame:
    """Replicate the XGBoost per-unit error breakdown."""
    error_df = eval_df.copy()
    error_df["tp"] = ((error_df["y_true"] == 1) & (error_df["y_pred"] == 1)).astype(int)
    error_df["tn"] = ((error_df["y_true"] == 0) & (error_df["y_pred"] == 0)).astype(int)
    error_df["fp"] = ((error_df["y_true"] == 0) & (error_df["y_pred"] == 1)).astype(int)
    error_df["fn"] = ((error_df["y_true"] == 1) & (error_df["y_pred"] == 0)).astype(int)

    spatial_error_df = (
        error_df.groupby(unit_col)
        .agg(
            n_samples=("y_true", "size"),
            true_count=("y_true", "sum"),
            pred_count=("y_pred", "sum"),
            tp_count=("tp", "sum"),
            tn_count=("tn", "sum"),
            fp_count=("fp", "sum"),
            fn_count=("fn", "sum"),
        )
        .reset_index()
    )
    spatial_error_df["fp_rate"] = spatial_error_df["fp_count"] / spatial_error_df["n_samples"]
    spatial_error_df["fn_rate"] = spatial_error_df["fn_count"] / spatial_error_df["n_samples"]
    return spatial_error_df


def run_texas_spatial_evaluation_for_target(
    target_col: str,
    task_name: str,
    model: RandomForestClassifier,
    ext_full: pd.DataFrame,
    ext_X: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Replicate the teammate XGBoost Texas spatial generalization analysis."""
    spatial_unit_col = get_texas_spatial_unit_col(ext_full)
    texas_eval_df = build_eval_df_for_target(spatial_unit_col, target_col, task_name, model, ext_full, ext_X)

    texas_spatial_eval_df = (
        texas_eval_df.groupby(spatial_unit_col)[["y_true", "y_pred", "y_prob"]]
        .apply(safe_group_metrics)
        .reset_index()
    )
    texas_spatial_eval_filtered = texas_spatial_eval_df[texas_spatial_eval_df["n_samples"] >= 10].copy()

    texas_unit_risk = (
        texas_eval_df.groupby(spatial_unit_col)["y_true"]
        .mean()
        .reset_index()
        .rename(columns={"y_true": "observed_risk"})
    )
    texas_spatial_eval_with_risk = texas_spatial_eval_filtered.merge(texas_unit_risk, on=spatial_unit_col, how="left")
    texas_spatial_eval_with_risk["risk_group"] = pd.qcut(
        texas_spatial_eval_with_risk["observed_risk"].rank(method="first"),
        q=3,
        labels=RISK_GROUP_ORDER,
    )

    texas_risk_group_summary = (
        texas_spatial_eval_with_risk.groupby("risk_group", observed=False)[["accuracy", "precision", "recall", "f1", "roc_auc"]]
        .mean()
        .reset_index()
    )
    texas_risk_group_summary["task"] = task_name

    texas_eval_df["task"] = task_name
    texas_spatial_eval_df["task"] = task_name
    texas_spatial_eval_filtered["task"] = task_name
    texas_spatial_eval_with_risk["task"] = task_name
    return texas_eval_df, texas_spatial_eval_df, texas_spatial_eval_filtered, texas_risk_group_summary


## 5. Plot Helpers
Create report-ready figures that mirror the structure of the latest XGBoost visuals.

In [ ]:
def plot_spatial_risk_group_summary(spatial_summary_df: pd.DataFrame, output_path: Path) -> None:
    """Create the one-row Chicago spatial comparison plot used in the report."""
    task_order = [task_name for _, task_name in TARGETS]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for ax, task_name in zip(axes, task_order):
        df_plot = spatial_summary_df[spatial_summary_df["task"] == task_name].copy()
        df_plot["risk_group"] = pd.Categorical(df_plot["risk_group"], categories=RISK_GROUP_ORDER, ordered=True)
        df_plot = df_plot.sort_values("risk_group")

        ax.bar(df_plot["risk_group"].astype(str), df_plot["f1"])
        ax.set_title(task_name, fontsize=12, fontweight="bold")
        ax.set_xlabel("Risk Group")
        ax.set_ylabel("Mean F1")
        ax.tick_params(axis="x", labelrotation=15)

    plt.suptitle("Spatial Accuracy Evaluation: F1 by Risk Group (Top 3 Crimes)", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(output_path, dpi=200)
    plt.close()


def plot_temporal_quarter_summary(quarter_summary_df: pd.DataFrame, output_path: Path) -> None:
    """Create the one-row quarterly temporal performance plot used in the report."""
    task_order = [task_name for _, task_name in TARGETS]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for ax, task_name in zip(axes, task_order):
        df_plot = quarter_summary_df[quarter_summary_df["task"] == task_name].copy()
        df_plot["quarter"] = pd.Categorical(df_plot["quarter"], categories=QUARTER_ORDER, ordered=True)
        df_plot = df_plot.sort_values("quarter")

        x = np.arange(len(df_plot))
        width = 0.35

        ax.bar(x - width / 2, df_plot["f1"], width, label="F1")
        ax.bar(x + width / 2, df_plot["roc_auc"], width, label="ROC-AUC")
        ax.set_xticks(x)
        ax.set_xticklabels(df_plot["quarter"].astype(str))
        ax.set_title(task_name, fontsize=12, fontweight="bold")
        ax.set_xlabel("Quarter")
        ax.set_ylabel("Score")

    axes[0].legend()
    plt.suptitle("Temporal Accuracy Measures: Quarterly F1 and ROC-AUC (Top 3 Crimes)", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(output_path, dpi=200)
    plt.close()


def plot_hotspot_summary(hotspot_summary_df: pd.DataFrame, output_path: Path) -> None:
    """Create the one-row hotspot hit-rate and coverage plot."""
    task_order = [task_name for _, task_name in TARGETS]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for ax, task_name in zip(axes, task_order):
        df_plot = hotspot_summary_df[hotspot_summary_df["task"] == task_name].copy()
        df_plot = df_plot.sort_values("k")

        ax.plot(df_plot["k"], df_plot["hit_rate"], marker="o", label="Hit Rate@k")
        ax.plot(df_plot["k"], df_plot["coverage"], marker="s", label="Coverage@k")
        ax.set_title(task_name, fontsize=12, fontweight="bold")
        ax.set_xlabel("Top-k grids")
        ax.set_ylabel("Score")
        ax.legend()

    plt.suptitle("Spatial Hotspot Evaluation for Top 3 Crimes", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(output_path, dpi=200)
    plt.close()


def plot_texas_risk_group_summary(texas_risk_summary_df: pd.DataFrame, output_path: Path) -> None:
    """Create the one-row Texas observed-risk grouping plot."""
    task_order = [task_name for _, task_name in TARGETS]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for ax, task_name in zip(axes, task_order):
        df_plot = texas_risk_summary_df[texas_risk_summary_df["task"] == task_name].copy()
        df_plot["risk_group"] = pd.Categorical(df_plot["risk_group"], categories=RISK_GROUP_ORDER, ordered=True)
        df_plot = df_plot.sort_values("risk_group")

        ax.bar(df_plot["risk_group"].astype(str), df_plot["f1"])
        ax.set_title(task_name, fontsize=12, fontweight="bold")
        ax.set_xlabel("Agency Risk Group")
        ax.set_ylabel("Mean F1")
        ax.tick_params(axis="x", labelrotation=15)

    plt.suptitle("Texas Generalization: High-risk vs Low-risk Spatial Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(output_path, dpi=200)
    plt.close()


def plot_feature_importance(top_feature_df: pd.DataFrame, output_path: Path) -> None:
    """Create a compact feature-importance figure for the report."""
    task_order = [task_name for _, task_name in TARGETS]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=False)

    for ax, task_name in zip(axes, task_order):
        df_plot = top_feature_df[top_feature_df["task"] == task_name].copy().sort_values("importance", ascending=True)
        ax.barh(df_plot["feature"], df_plot["importance"])
        ax.set_title(task_name, fontsize=12, fontweight="bold")
        ax.set_xlabel("Importance")

    plt.suptitle("Random Forest Top Feature Importances (Top 3 Crimes)", fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(output_path, dpi=200)
    plt.close()


## 6. Main Execution and Export
Run the full RF workflow and export all tables, figures, and summary payloads for the reports.

In [ ]:
def main() -> None:
    """Run the full RF pipeline aligned with the team's latest XGBoost structure."""
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    train_X = pd.read_csv(CHICAGO_TRAIN_X)
    train_full = pd.read_csv(CHICAGO_TRAIN_FULL)
    test_X = pd.read_csv(CHICAGO_TEST_X)
    test_full = pd.read_csv(CHICAGO_TEST_FULL)
    ext_X = pd.read_csv(TEXAS_X)
    ext_full = pd.read_csv(TEXAS_FULL)

    cv_results_all: List[pd.DataFrame] = []
    cv_summary_all: List[pd.DataFrame] = []
    test_summary_rows: List[Dict[str, object]] = []
    top_feature_rows: List[Dict[str, object]] = []

    chicago_prediction_df = test_full[["grid_id", "iso_year", "iso_week", "year_week"]].copy()
    texas_unit_col = get_texas_spatial_unit_col(ext_full)
    texas_prediction_df = ext_full[[texas_unit_col, "iso_year", "iso_week", "year_week"]].copy()

    chicago_spatial_eval_all: List[pd.DataFrame] = []
    chicago_spatial_summary_all: List[pd.DataFrame] = []
    chicago_temporal_week_all: List[pd.DataFrame] = []
    chicago_temporal_quarter_all: List[pd.DataFrame] = []
    chicago_hotspot_all: List[pd.DataFrame] = []
    chicago_spatial_error_all: List[pd.DataFrame] = []

    texas_spatial_eval_all: List[pd.DataFrame] = []
    texas_risk_summary_all: List[pd.DataFrame] = []

    summary_payload: Dict[str, Dict[str, object]] = {}

    for target_col, task_name in TARGETS:
        cv_results_df, cv_summary_df = run_timeseries_cv_for_target(train_full, train_X, target_col, task_name, n_splits=5)
        cv_results_all.append(cv_results_df)
        cv_summary_all.append(cv_summary_df)

        output = run_rf_for_one_target(
            target_col=target_col,
            task_name=task_name,
            train_full=train_full,
            train_X=train_X,
            test_full=test_full,
            test_X=test_X,
            ext_full=ext_full,
            ext_X=ext_X,
        )
        model = output["model"]

        feature_importance_df = pd.DataFrame(
            {
                "task": task_name,
                "feature": train_X.columns,
                "importance": model.feature_importances_,
            }
        ).sort_values("importance", ascending=False)
        for row in feature_importance_df.head(10).to_dict(orient="records"):
            top_feature_rows.append(row)

        test_summary_rows.append(
            {
                "task": task_name,
                "split": "Chicago2025",
                "model": "Random Forest",
                "accuracy": output["chicago_result"]["accuracy"],
                "precision": output["chicago_result"]["precision"],
                "recall": output["chicago_result"]["recall"],
                "f1": output["chicago_result"]["f1"],
                "auc": output["chicago_result"]["auc"],
            }
        )
        test_summary_rows.append(
            {
                "task": task_name,
                "split": "TexasExternal",
                "model": "Random Forest",
                "accuracy": output["texas_result"]["accuracy"],
                "precision": output["texas_result"]["precision"],
                "recall": output["texas_result"]["recall"],
                "f1": output["texas_result"]["f1"],
                "auc": output["texas_result"]["auc"],
            }
        )

        task_slug = TASK_SLUG_BY_TARGET[target_col]
        chicago_prediction_df[f"pred_{task_slug}"] = output["y_test_pred"]
        chicago_prediction_df[f"prob_{task_slug}"] = output["y_test_prob"]
        texas_prediction_df[f"pred_{task_slug}"] = output["y_ext_pred"]
        texas_prediction_df[f"prob_{task_slug}"] = output["y_ext_prob"]

        chicago_eval_df, chicago_spatial_eval_df, _chicago_spatial_with_risk_df, chicago_risk_summary_df = run_spatial_evaluation_for_target(
            target_col=target_col,
            task_name=task_name,
            model=model,
            train_full=train_full,
            test_full=test_full,
            test_X=test_X,
        )
        chicago_spatial_eval_all.append(chicago_spatial_eval_df)
        chicago_spatial_summary_all.append(chicago_risk_summary_df)

        _hotspot_grid_summary_df, hotspot_metrics_df = compute_hotspot_metrics(
            chicago_eval_df,
            unit_col="grid_id",
            k_list=HOTSPOT_K_LIST,
        )
        hotspot_metrics_df["task"] = task_name
        chicago_hotspot_all.append(hotspot_metrics_df)

        chicago_spatial_error_df = compute_spatial_error_by_unit(chicago_eval_df, unit_col="grid_id")
        chicago_spatial_error_df["task"] = task_name
        chicago_spatial_error_all.append(chicago_spatial_error_df)

        temporal_week_df, temporal_quarter_df = run_temporal_evaluation_for_target(
            target_col=target_col,
            task_name=task_name,
            model=model,
            test_full=test_full,
            test_X=test_X,
        )
        chicago_temporal_week_all.append(temporal_week_df)
        chicago_temporal_quarter_all.append(temporal_quarter_df)

        _texas_eval_df, _texas_spatial_eval_df, texas_spatial_eval_filtered_df, texas_risk_summary_df = run_texas_spatial_evaluation_for_target(
            target_col=target_col,
            task_name=task_name,
            model=model,
            ext_full=ext_full,
            ext_X=ext_X,
        )
        texas_spatial_eval_all.append(texas_spatial_eval_filtered_df)
        texas_risk_summary_all.append(texas_risk_summary_df)

        summary_payload[task_name] = {
            "cv_summary": cv_summary_df.to_dict(orient="records"),
            "chicago_result": output["chicago_result"],
            "texas_result": output["texas_result"],
            "feature_importance_top10": feature_importance_df.head(10).to_dict(orient="records"),
            "chicago_risk_summary": chicago_risk_summary_df.to_dict(orient="records"),
            "temporal_quarter_summary": temporal_quarter_df.to_dict(orient="records"),
            "hotspot_summary": hotspot_metrics_df.to_dict(orient="records"),
            "texas_risk_summary": texas_risk_summary_df.to_dict(orient="records"),
        }

    cv_results_df = pd.concat(cv_results_all, ignore_index=True)
    cv_summary_df = pd.concat(cv_summary_all, ignore_index=True)
    test_summary_df = pd.DataFrame(test_summary_rows)
    top_feature_df = pd.DataFrame(top_feature_rows)
    chicago_spatial_eval_df = pd.concat(chicago_spatial_eval_all, ignore_index=True)
    chicago_spatial_summary_df = pd.concat(chicago_spatial_summary_all, ignore_index=True)
    chicago_temporal_week_df = pd.concat(chicago_temporal_week_all, ignore_index=True)
    chicago_temporal_quarter_df = pd.concat(chicago_temporal_quarter_all, ignore_index=True)
    chicago_hotspot_df = pd.concat(chicago_hotspot_all, ignore_index=True)
    chicago_spatial_error_df = pd.concat(chicago_spatial_error_all, ignore_index=True)
    texas_spatial_eval_df = pd.concat(texas_spatial_eval_all, ignore_index=True)
    texas_risk_summary_df = pd.concat(texas_risk_summary_all, ignore_index=True)

    cv_results_df.to_csv(OUTPUT_DIR / "rf_cv_results.csv", index=False)
    cv_summary_df.to_csv(OUTPUT_DIR / "rf_cv_summary.csv", index=False)
    test_summary_df.to_csv(OUTPUT_DIR / "rf_test_summary.csv", index=False)
    top_feature_df.to_csv(OUTPUT_DIR / "rf_top_feature_importance.csv", index=False)
    chicago_prediction_df.to_csv(OUTPUT_DIR / "rf_chicago_top3_predictions.csv", index=False)
    texas_prediction_df.to_csv(OUTPUT_DIR / "rf_texas_top3_predictions.csv", index=False)
    chicago_spatial_eval_df.to_csv(OUTPUT_DIR / "rf_top3_chicago_spatial_unit_metrics.csv", index=False)
    chicago_spatial_summary_df.to_csv(OUTPUT_DIR / "rf_top3_spatial_summary.csv", index=False)
    chicago_temporal_week_df.to_csv(OUTPUT_DIR / "rf_top3_temporal_weekly_summary.csv", index=False)
    chicago_temporal_quarter_df.to_csv(OUTPUT_DIR / "rf_top3_temporal_quarter_summary.csv", index=False)
    chicago_hotspot_df.to_csv(OUTPUT_DIR / "rf_top3_hotspot_summary.csv", index=False)
    chicago_spatial_error_df.to_csv(OUTPUT_DIR / "rf_top3_spatial_error_by_grid.csv", index=False)
    texas_spatial_eval_df.to_csv(OUTPUT_DIR / "rf_top3_texas_spatial_unit_metrics.csv", index=False)
    texas_risk_summary_df.to_csv(OUTPUT_DIR / "rf_top3_texas_risk_summary.csv", index=False)

    plot_spatial_risk_group_summary(chicago_spatial_summary_df, OUTPUT_DIR / "rf_top3_spatial_summary.png")
    plot_temporal_quarter_summary(chicago_temporal_quarter_df, OUTPUT_DIR / "rf_top3_temporal_quarter_summary.png")
    plot_hotspot_summary(chicago_hotspot_df, OUTPUT_DIR / "rf_top3_hotspot_summary.png")
    plot_texas_risk_group_summary(texas_risk_summary_df, OUTPUT_DIR / "rf_top3_texas_risk_summary.png")
    plot_feature_importance(top_feature_df, OUTPUT_DIR / "rf_top_feature_importance.png")

    with (OUTPUT_DIR / "rf_summary_payload.json").open("w", encoding="utf-8") as fp:
        json.dump(summary_payload, fp, indent=2)

    print("RF V4 TimeSeriesSplit CV summary:")
    print(cv_summary_df.to_string(index=False))
    print()
    print("RF V4 Chicago/Texas test summary:")
    print(test_summary_df.to_string(index=False))
    print()
    print("Outputs written to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()
